In [1]:
from agents import Agent, trace, WebSearchTool, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv
import resend
import asyncio
from typing import Dict
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)

True

In [3]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_Agent= Agent(
    name="search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required")
    
)

In [6]:
message= "Letest ai framwork in 2026"

with trace("search"):
    result= await Runner.run(search_Agent, message)
    
display(Markdown(result.final_output))

In 2026, several AI frameworks have emerged, each addressing specific challenges in the field:

- **Orchestral AI**: A Python framework offering a unified, type-safe interface for building LLM agents across major providers, aiming to simplify agent orchestration and enhance reproducibility. ([arxiv.org](https://arxiv.org/abs/2601.02577?utm_source=openai))

- **HyperParallel**: Developed within MindSpore, this framework optimizes AI models for supernode architectures, integrating hundreds to thousands of accelerators with ultra-low-latency interconnects and unified memory pools. ([arxiv.org](https://arxiv.org/abs/2603.03731?utm_source=openai))

- **HAIF (Human-AI Integration Framework)**: A protocol-based system designed to integrate AI agents into human teams, emphasizing delegation, autonomy, and feedback mechanisms to enhance collaboration. ([arxiv.org](https://arxiv.org/abs/2602.07641?utm_source=openai))

- **Agent2Agent (A2A)**: An open protocol for AI agent communication, enabling interoperability across different systems and facilitating task coordination among agents from various vendors. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Agent2Agent?utm_source=openai))

- **CSA AI Controls Framework**: A vendor-neutral framework tailored for cloud-based AI environments, providing a detailed roadmap of control objectives across 18 security domains to ensure safe AI implementation. ([cycoresecure.com](https://www.cycoresecure.com/blogs/best-ai-security-frameworks-organizations-2026?utm_source=openai))

- **LlamaIndex**: Evolved from a data framework, it now offers a platform for building agents deeply integrated with proprietary data sources and knowledge bases, featuring advanced indexing and retrieval pipelines. ([aitude.com](https://www.aitude.com/top-agentic-ai-frameworks-2026/?utm_source=openai))

- **Semantic Kernel**: Microsoft's enterprise-grade AI orchestration framework supporting multiple languages and designed for robust governance, security, and integration with existing business systems. ([aitude.com](https://www.aitude.com/top-agentic-ai-frameworks-2026/?utm_source=openai))

These frameworks reflect the industry's focus on enhancing AI agent interoperability, security, and integration within enterprise environments. 

In [8]:
HOW_MANY_SEARCHES= 3

instruction= "You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."


class websearchitem(BaseModel):
    resone: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")
    
class websearchplan(BaseModel):
    searches: list[websearchitem] = Field(description="A list of web searches to perform to best answer the query.")
    
    
planner_agent= Agent(
    name="planner_agent",
    instructions=instruction,
    output_type=websearchplan,
    model="gpt-4o-mini"
)
    


In [11]:
message= "Lestest AI framwork in 2026"

with trace("searches"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)
    

searches=[websearchitem(resone='To find the latest AI frameworks available in 2026, focusing on technological advancements and emerging tools.', query='latest AI frameworks 2026'), websearchitem(resone='To understand industry trends and popular AI frameworks that are gaining traction in 2026.', query='popular AI frameworks 2026'), websearchitem(resone='To gather insights from authoritative sources about predictions and developments in AI frameworks by 2026.', query='AI framework development predictions 2026'), websearchitem(resone='To identify key players and organizations leading the creation of new AI frameworks in 2026.', query='top AI framework companies 2026')]


In [ ]:
plan = await Runner.run(planner_agent, message)
for item in plan.final_output.searches:
    result = await Runner.run(search_Agent, item.query)
    print(item.query, "->", result.final_output)

In [17]:
@function_tool
def send_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body to all sales prospects using Resend"""
    
    from_email = "onboarding@resend.dev"
    to_email = "shahid739815@gmail.com"
    
    resend.api_key = os.environ.get("RESEND_API_KEY")
    
    print(f"Sending email...")
    print(f"From: {from_email}")
    print(f"To: {to_email}")
    print(f"Subject: {subject}")
    print(f"API Key present: {bool(resend.api_key)}")
    
    try:
        params = {
            "from": from_email,
            "to": [to_email],
            "subject": subject,
            "html": html_body
        }
        
        email = resend.Emails.send(params)
        print(f"Response: {email}")
        
        if email.get("id"):
            return f"Email sent successfully with ID: {email['id']}"
        else:
            return f"Email failed to send: {email}"
    except Exception as e:
        print(f"Error: {e}")
        return f"Email failed to send due to error: {str(e)}"

In [19]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body to all sales prospects using Resend', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000028C016CD120>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [20]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

send_email= Agent(
    name="send_email",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model= "gpt-4o-mini"
)

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class reportdata(BaseModel):
    short_summery: str = Field(description="A short 2-3 sentence summary of the findings.")
    
    markdown_report: str = Field(description="The final report")
    
    folloup_questions= list(str) = Field(description="Suggested topics to research further")
    
    
writer_agent= Agent(
    name="writer_agent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=reportdata
)

In [ ]:
fruits = ["Apple", "Banana", "Cherry"]

#print(fruits)
print(*fruits)

Apple Banana Cherry


: 